In [0]:
%sql
-- Creating a catalog and schema 

Create catalog if not exists cdc_catalog;
create schema if not exists cdc_catalog.cdc_schema; 
create volume if not exists cdc_catalog.cdc_schema.cdc_customer_vol_ckpoint;
use  cdc_catalog.cdc_schema;



In [0]:
#Checking the path 

print('The below path will drop the checkout path...by default it will be disabled')
chkpnt_path = '/Volumes/dbx_catalog/dbx_schema/cdc_customer_vol_ckpoint'
print(f"The checkpoint path :-------- {chkpnt_path}")
#dbutils.fs.rm(f"{chkpnt_path}",recurse=True)

The below path will drop the checkout path...by default it will be disabled
The checkpoint path :-------- /Volumes/dbx_catalog/dbx_schema/cdf_intermediate_checkpoint


In [0]:
%sql
/*
DROP TABLE customer_source_table;
*/

Create table customer_source_table(
  userId  INT,name STRING , city STRING)
  USING DELTA 
  TBLPROPERTIES (delta.enableChangeDataFeed = true,
                 delta.deletedFileRetentionDuration = 'interval 1 days');

/* incase customer_base_table already created
ALTER TABLE customer_base_table 
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);
*/


In [0]:
%sql
DESCRIBE TABLE EXTENDED customer_source_table


col_name,data_type,comment
userId,int,null
name,string,null
city,string,null
,,
# Detailed Table Information,,
Catalog,cdc_catalog,
Database,cdc_schema,
Table,customer_source_table,
Created Time,Wed Jul 15 09:10:41 UTC 2026,
Last Access,UNKNOWN,


In [0]:
%sql
 SELECT * 
   FROM table_changes("customer_source_table", 0)

userId,name,city,_change_type,_commit_version,_commit_timestamp


In [0]:
%sql
Insert into customer_source_table
VALUES (101, "Raul", "Oaxaca")

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
 SELECT * 
   FROM table_changes("customer_source_table",0 )

userId,name,city,_change_type,_commit_version,_commit_timestamp
101,Raul,Oaxaca,insert,1,2026-07-15T09:12:31.000Z


In [0]:
%sql
VACUUM customer_base_table;

path
""


In [0]:
%sql
select * from (DESCRIBE HISTORY customer_source_table) order by version desc


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-07-15T09:24:34.000Z,7912518962361895,sandutta2020@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(534018730120359),433f5cb0-dffa-4430-991a-fa4fc183395c,0715-090242-b8i6vgma-v2n,1,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1230)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
1,2026-07-15T09:12:31.000Z,7912518962361895,sandutta2020@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(534018730120359),ec1879d4-13d9-4b3b-b5d4-69d42e07d5ac,0715-090242-b8i6vgma-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1230)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
0,2026-07-15T09:10:41.000Z,7912518962361895,sandutta2020@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableChangeDataFeed"":""true"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-a914242f-7724-4440-a566-7ff0f25e458c"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-c51ef6cf-7145-4420-af2d-c1ee8e4741cb"",""delta.deletedFileRetentionDuration"":""interval 1 days"",""delta.parquet.format.version.afe.internal"":""2.12.0""}, statsOnLoad -> false)",null,List(534018730120359),3d71773b-9853-4576-9cc9-6acc279a73a4,0715-090242-b8i6vgma-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/18.x-aarch64-photon-scala2.13


In [0]:
%sql
insert into customer_source_table VALUES
(102, "Jhon", "Mexico")

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
 SELECT * 
   FROM table_changes("customer_source_table", 0)

userId,name,city,_change_type,_commit_version,_commit_timestamp
102,Jhon,Mexico,insert,2,2026-07-15T09:24:34.000Z
101,Raul,Oaxaca,insert,1,2026-07-15T09:12:31.000Z


In [0]:
%sql
DESCRIBE HISTORY customer_source_table


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-07-15T09:24:34.000Z,7912518962361895,sandutta2020@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(534018730120359),433f5cb0-dffa-4430-991a-fa4fc183395c,0715-090242-b8i6vgma-v2n,1,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1230)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
1,2026-07-15T09:12:31.000Z,7912518962361895,sandutta2020@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(534018730120359),ec1879d4-13d9-4b3b-b5d4-69d42e07d5ac,0715-090242-b8i6vgma-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1230)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
0,2026-07-15T09:10:41.000Z,7912518962361895,sandutta2020@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableChangeDataFeed"":""true"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-a914242f-7724-4440-a566-7ff0f25e458c"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-c51ef6cf-7145-4420-af2d-c1ee8e4741cb"",""delta.deletedFileRetentionDuration"":""interval 1 days"",""delta.parquet.format.version.afe.internal"":""2.12.0""}, statsOnLoad -> false)",null,List(534018730120359),3d71773b-9853-4576-9cc9-6acc279a73a4,0715-090242-b8i6vgma-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/18.x-aarch64-photon-scala2.13


In [0]:
%sql
insert into customer_source_table VALUES
(103, "Lily", "Canada"),
(104, "colin", "USA"),
(105, "mike", "Colombo") ;
delete from customer_source_table where userId = 102;

num_affected_rows
1


In [0]:
%sql
select * from customer_source_table

userId,name,city
103,Lily,Canada
104,colin,USA
105,mike,Colombo
101,Raul,Oaxaca


In [0]:
%sql
 SELECT * 
   FROM table_changes("customer_source_table", 0)

userId,name,city,_change_type,_commit_version,_commit_timestamp
103,Lily,Canada,insert,3,2026-07-15T10:49:26.000Z
104,colin,USA,insert,3,2026-07-15T10:49:26.000Z
105,mike,Colombo,insert,3,2026-07-15T10:49:26.000Z
102,Jhon,Mexico,insert,2,2026-07-15T09:24:34.000Z
101,Raul,Oaxaca,insert,1,2026-07-15T09:12:31.000Z
102,Jhon,Mexico,delete,4,2026-07-15T10:49:30.000Z


In [0]:
%sql
---updating a table

update customer_source_table set city='New York' where userID=104;

num_affected_rows
1


In [0]:
%sql
 SELECT * 
   FROM table_changes("customer_source_table", 0)

userId,name,city,_change_type,_commit_version,_commit_timestamp
104,colin,USA,update_preimage,5,2026-07-15T10:53:51.000Z
104,colin,New York,update_postimage,5,2026-07-15T10:53:51.000Z
103,Lily,Canada,insert,3,2026-07-15T10:49:26.000Z
104,colin,USA,insert,3,2026-07-15T10:49:26.000Z
105,mike,Colombo,insert,3,2026-07-15T10:49:26.000Z
102,Jhon,Mexico,insert,2,2026-07-15T09:24:34.000Z
101,Raul,Oaxaca,insert,1,2026-07-15T09:12:31.000Z
102,Jhon,Mexico,delete,4,2026-07-15T10:49:30.000Z


In [0]:
%sql
---updating a table

update customer_source_table set city='Florida' where userID=104;

num_affected_rows
1


In [0]:
%sql
 SELECT * 
   FROM table_changes("customer_source_table", 0)

userId,name,city,_change_type,_commit_version,_commit_timestamp
104,colin,New York,update_preimage,7,2026-07-15T10:55:43.000Z
104,colin,Florida,update_postimage,7,2026-07-15T10:55:43.000Z
104,colin,USA,update_preimage,5,2026-07-15T10:53:51.000Z
104,colin,New York,update_postimage,5,2026-07-15T10:53:51.000Z
103,Lily,Canada,insert,3,2026-07-15T10:49:26.000Z
104,colin,USA,insert,3,2026-07-15T10:49:26.000Z
105,mike,Colombo,insert,3,2026-07-15T10:49:26.000Z
102,Jhon,Mexico,insert,2,2026-07-15T09:24:34.000Z
101,Raul,Oaxaca,insert,1,2026-07-15T09:12:31.000Z
102,Jhon,Mexico,delete,4,2026-07-15T10:49:30.000Z


In [0]:
"""cdf_df = (spark.readStream
               .format("delta")
               .option("readChangeData", True)
               .option("startingVersion", 5)
               .table("customer_table"))

display(cdf_df, checkpointLocation = "/Volumes/dbx_catalog/dbx_schema/cdf_checkpoint")"""

spark.\
    readStream.\
        option("readChangeFeed", "true").\
            option("startingVersion", 0).\
                table("customer_source_table").\
                    writeStream.\
                        option("checkpointLocation", "/Volumes/cdc_catalog/cdc_schema/cdc_customer_vol_ckpoint").\
                            trigger (availableNow=True).\
                                table("customers_cdf")


In [0]:
%sql
select * from customers_cdf

userId,name,city,_change_type,_commit_version,_commit_timestamp
104,colin,New York,update_preimage,7,2026-07-15T10:55:43.000Z
104,colin,Florida,update_postimage,7,2026-07-15T10:55:43.000Z
104,colin,USA,update_preimage,5,2026-07-15T10:53:51.000Z
104,colin,New York,update_postimage,5,2026-07-15T10:53:51.000Z
103,Lily,Canada,insert,3,2026-07-15T10:49:26.000Z
104,colin,USA,insert,3,2026-07-15T10:49:26.000Z
105,mike,Colombo,insert,3,2026-07-15T10:49:26.000Z
101,Raul,Oaxaca,insert,1,2026-07-15T09:12:31.000Z
102,Jhon,Mexico,insert,2,2026-07-15T09:24:34.000Z
102,Jhon,Mexico,delete,4,2026-07-15T10:49:30.000Z


In [0]:
%sql
update customer_source_table set city='washington DC' where userId=103

num_affected_rows
1


In [0]:
%sql
 SELECT * 
   FROM table_changes("customer_source_table", 0)

userId,name,city,_change_type,_commit_version,_commit_timestamp
103,Lily,Canada,update_preimage,9,2026-07-15T11:14:57.000Z
103,Lily,washington DC,update_postimage,9,2026-07-15T11:14:57.000Z
104,colin,New York,update_preimage,7,2026-07-15T10:55:43.000Z
104,colin,Florida,update_postimage,7,2026-07-15T10:55:43.000Z
104,colin,USA,update_preimage,5,2026-07-15T10:53:51.000Z
104,colin,New York,update_postimage,5,2026-07-15T10:53:51.000Z
103,Lily,Canada,insert,3,2026-07-15T10:49:26.000Z
104,colin,USA,insert,3,2026-07-15T10:49:26.000Z
105,mike,Colombo,insert,3,2026-07-15T10:49:26.000Z
102,Jhon,Mexico,insert,2,2026-07-15T09:24:34.000Z


In [0]:
from pyspark.sql.functions import col
spark.\
    readStream.\
        option("readChangeFeed", "true").\
            option("startingVersion", 0).\
                table("customer_source_table").\
                    filter(col("_change_type").isin(["update_postimage"])).\
                        writeStream.\
                            option("checkpointLocation", "/Volumes/cdc_catalog/cdc_schema/cdc_customer_vol_ckpoint").\
                                trigger (availableNow=True).\
                                    table("customers_cdf")

In [0]:
%sql
select * from customers_cdf;

userId,name,city,_change_type,_commit_version,_commit_timestamp
104,colin,New York,update_preimage,7,2026-07-15T10:55:43.000Z
104,colin,Florida,update_postimage,7,2026-07-15T10:55:43.000Z
104,colin,USA,update_preimage,5,2026-07-15T10:53:51.000Z
104,colin,New York,update_postimage,5,2026-07-15T10:53:51.000Z
103,Lily,Canada,insert,3,2026-07-15T10:49:26.000Z
104,colin,USA,insert,3,2026-07-15T10:49:26.000Z
105,mike,Colombo,insert,3,2026-07-15T10:49:26.000Z
101,Raul,Oaxaca,insert,1,2026-07-15T09:12:31.000Z
102,Jhon,Mexico,insert,2,2026-07-15T09:24:34.000Z
103,Lily,washington DC,update_postimage,9,2026-07-15T11:14:57.000Z


In [0]:
%sql
update customer_source_table set city='Miami' where userId=103

num_affected_rows
1


In [0]:
%sql
 SELECT * 
   FROM table_changes("customer_source_table", 0)

userId,name,city,_change_type,_commit_version,_commit_timestamp
103,Lily,Canada,update_preimage,9,2026-07-15T11:14:57.000Z
103,Lily,washington DC,update_postimage,9,2026-07-15T11:14:57.000Z
104,colin,New York,update_preimage,7,2026-07-15T10:55:43.000Z
104,colin,Florida,update_postimage,7,2026-07-15T10:55:43.000Z
103,Lily,washington DC,update_preimage,11,2026-07-15T11:22:48.000Z
103,Lily,Miami,update_postimage,11,2026-07-15T11:22:48.000Z
104,colin,USA,update_preimage,5,2026-07-15T10:53:51.000Z
104,colin,New York,update_postimage,5,2026-07-15T10:53:51.000Z
103,Lily,Canada,insert,3,2026-07-15T10:49:26.000Z
104,colin,USA,insert,3,2026-07-15T10:49:26.000Z


In [0]:
from pyspark.sql.functions import col
spark.\
    readStream.\
        option("readChangeFeed", "true").\
            option("startingVersion", 0).\
                table("customer_source_table").\
                    filter(col("_change_type").isin(["update_postimage"])).\
                        writeStream.\
                            option("checkpointLocation", "/Volumes/cdc_catalog/cdc_schema/cdc_customer_vol_ckpoint").\
                                    trigger (availableNow=True).\
                                        table("customers_cdf")

In [0]:
%sql
select * from customers_cdf

userId,name,city,_change_type,_commit_version,_commit_timestamp
104,colin,New York,update_preimage,7,2026-07-15T10:55:43.000Z
104,colin,Florida,update_postimage,7,2026-07-15T10:55:43.000Z
104,colin,USA,update_preimage,5,2026-07-15T10:53:51.000Z
104,colin,New York,update_postimage,5,2026-07-15T10:53:51.000Z
103,Lily,Canada,insert,3,2026-07-15T10:49:26.000Z
104,colin,USA,insert,3,2026-07-15T10:49:26.000Z
105,mike,Colombo,insert,3,2026-07-15T10:49:26.000Z
101,Raul,Oaxaca,insert,1,2026-07-15T09:12:31.000Z
102,Jhon,Mexico,insert,2,2026-07-15T09:24:34.000Z
103,Lily,washington DC,update_postimage,9,2026-07-15T11:14:57.000Z
